# Loopy v2 Colab Baseline

This notebook reproduces the current best CPU baseline for Loopy v2 on GPU, measures the learned bitstream size, and then runs a small `rate_weight=0.001` comparison.

Repo: `darwesh88/googlereview`
Dataset: `loopy/data/real/twitter_support_5k.txt`

If the repo is private, the first code cell will prompt for a GitHub token with read access.


In [ ]:
import os
import shutil
import subprocess
from getpass import getpass

repo_dir = '/content/googlereview'
repo_url = 'https://github.com/darwesh88/googlereview.git'

if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

def clone_repo(url):
    subprocess.run(['git', 'clone', url, repo_dir], check=True)

try:
    clone_repo(repo_url)
except subprocess.CalledProcessError:
    token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not token:
        token = getpass('GitHub token with repo read access: ').strip()
    auth_url = repo_url.replace('https://', f'https://{token}@')
    clone_repo(auth_url)

os.chdir(repo_dir)
print('Cloned into', os.getcwd())
print('Top-level files:')
print('\n'.join(sorted(os.listdir('.'))[:20]))


In [ ]:
import os
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
req_path = os.path.join('loopy', 'requirements.txt')
if os.path.exists(req_path):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', req_path], check=True)
else:
    print('No loopy/requirements.txt found, continuing')


In [ ]:
import os
import torch

print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))

dataset_path = 'loopy/data/real/twitter_support_5k.txt'
print('dataset exists', os.path.exists(dataset_path), dataset_path)


## Baseline run (`rate_weight=0.0`)


In [ ]:
!python -m loopy.train_binary_codec_v2 \
  --data-path loopy/data/real/twitter_support_5k.txt \
  --output-dir loopy/runs/v2_colab_baseline \
  --epochs 20 \
  --batch-size 16 \
  --max-seq-len 128 \
  --patch-size 4 \
  --embed-dim 128 \
  --latent-dim 128 \
  --encoder-layers 2 \
  --decoder-layers 2 \
  --num-heads 4 \
  --dropout 0.0 \
  --weight-decay 0.0 \
  --bit-groups 6,6,6,6 \
  --rate-weight 0.0 \
  --balance-weight 0.001 \
  --align-weight 0.05


In [ ]:
!cat loopy/runs/v2_colab_baseline/best_metrics.json
print()
!cat loopy/runs/v2_colab_baseline/sample_reconstruction.txt


In [ ]:
!python -m loopy.measure_bitstream_v2 \
  --run-dir loopy/runs/v2_colab_baseline \
  --data-path loopy/data/real/twitter_support_5k.txt \
  --output loopy/runs/v2_colab_baseline/bitstream_summary.json
!cat loopy/runs/v2_colab_baseline/bitstream_summary.json


## Rate-aware comparison (`rate_weight=0.001`)

Run this after the baseline so the two outputs stay side by side.


In [ ]:
!python -m loopy.train_binary_codec_v2 \
  --data-path loopy/data/real/twitter_support_5k.txt \
  --output-dir loopy/runs/v2_colab_rate_small \
  --epochs 20 \
  --batch-size 16 \
  --max-seq-len 128 \
  --patch-size 4 \
  --embed-dim 128 \
  --latent-dim 128 \
  --encoder-layers 2 \
  --decoder-layers 2 \
  --num-heads 4 \
  --dropout 0.0 \
  --weight-decay 0.0 \
  --bit-groups 6,6,6,6 \
  --rate-weight 0.001 \
  --balance-weight 0.001 \
  --align-weight 0.05


In [ ]:
!cat loopy/runs/v2_colab_rate_small/best_metrics.json
print()
!cat loopy/runs/v2_colab_rate_small/sample_reconstruction.txt


In [ ]:
!python -m loopy.measure_bitstream_v2 \
  --run-dir loopy/runs/v2_colab_rate_small \
  --data-path loopy/data/real/twitter_support_5k.txt \
  --output loopy/runs/v2_colab_rate_small/bitstream_summary.json
!cat loopy/runs/v2_colab_rate_small/bitstream_summary.json
